# Estrazione Dataset: Formazione vs Implementazione

Questo notebook utilizza uno script che sfrutta la libreria `multiprocessing` per estrarre 10.000 campioni unici di `DESCRIZIONE_PROGETTO` dai dataset in `data/raw`, etichettandoli (in base al campo `OBIETTIVO`) in due categorie:
- `formazione`
- `implementazione`

In [15]:
%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd

# Aggiungo src/ al path per poter importare lo script
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src')))

try:
    from regex_multiprocessing import run_extraction
except ImportError:
    print("Assicurati di aver creato run_extraction in src/regex_multiprocessing.py")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Esecuzione Estrazione (Multiprocessing)
Avvio il processo di estrazione ricercando fino a 10.000 campioni

In [16]:
data_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'data', 'raw'))
print("Leggo i file da:", data_dir)

# Estrazione
df_dataset = run_extraction(data_dir=data_dir, target_per_class=5000)

print(f"Dimensione dataset estratto: {len(df_dataset)}")
print(df_dataset["label"].value_counts())

Leggo i file da: /home/gabs/Documenti/Università/AI nelle Imprese/open-data-analytics/data/raw
Trovati 12 file CSV da processare.
Campioni aggregati - Formazione: 2, Implementazione: 312
Campioni aggregati - Formazione: 32, Implementazione: 707
Campioni aggregati - Formazione: 32, Implementazione: 977
Campioni aggregati - Formazione: 3806, Implementazione: 4715
Campioni aggregati - Formazione: 20376, Implementazione: 11794
Raggiunto il target desiderato per entrambe le classi!
Dimensione dataset estratto: 10000
label
implementazione    5000
formazione         5000
Name: count, dtype: int64


## Salvataggio Dataset
Salviamo il dataframe generato in un file CSV sotto `data/distilled`

In [17]:
output_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'data', 'distilled'))
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, 'dataset_formazione_implementazione.csv')

df_dataset.to_csv(output_file, index=False)
print(f"Salvato in {output_file}")

Salvato in /home/gabs/Documenti/Università/AI nelle Imprese/open-data-analytics/data/distilled/dataset_formazione_implementazione.csv


In [18]:
# Visualizza qualche sample
df_dataset.head()

,DESCRIZIONE_PROGETTO,label
0,AttivitÃ¿Â¿Ã¿Â per la diffusione di una migli...,implementazione
1,AA10 CORRETTO UTILIZZO DEL CRONOTACHIGRAFO E C...,formazione
2,Il percorso formativo proposto puo essere cons...,formazione
3,Internazionalizzazione dei mercati nell'impres...,formazione
4,L'innovazione tecnologica che abilita l'innova...,formazione


## Estrazione Goldset AI (formazione vs implementazione)
Estraiamo un ulteriore dataset che contiene le descrizioni (distinte tra formazione e implementazione) unicamente per i record che hanno `CLASSIFICAZIONE` uguale a "AI". Il file verrà poi salvato in `data/distilled/goldset_form_impl.csv`.

In [19]:
from regex_multiprocessing import run_ai_extraction

# Estrazione goldset AI
df_goldset_ai = run_ai_extraction(data_dir=data_dir, target_per_class=100)

print(f"Dimensione dataset estratto AI: {len(df_goldset_ai)}")
print(df_goldset_ai["label"].value_counts())

# Salvataggio Dataset AI
output_file_ai = os.path.join(output_dir, 'goldset_form_impl.csv')
df_goldset_ai.to_csv(output_file_ai, index=False)
print(f"Salvato in {output_file_ai}")

df_goldset_ai.head()

ImportError: cannot import name 'run_ai_extraction' from 'regex_multiprocessing' (/home/gabs/Documenti/Università/AI nelle Imprese/open-data-analytics/src/regex_multiprocessing.py)

## Classificazione estesa a tutto il dataset (TIPO_AI)
Estendiamo la classificazione calcolata per "formazione" e "implementazione" a tutto il dataset originale nei file presenti nella directory `raw`. Per ogni file, i record classificati come "AI" (nella colonna `CLASSIFICAZIONE`) verranno processati dalle regex e la nuova etichetta verrà inserita in-place nella colonna `TIPO_AI`.

In [20]:
from regex_multiprocessing import update_all_raw_files

# Estendi la classificazione TIPO_AI su tutti i file raw (solo ai record con classificazione 'AI')
update_all_raw_files(data_dir)

print("Estensione classificazione TIPO_AI completata!")

Aggiornamento TIPO_AI su 12 file CSV nella directory /home/gabs/Documenti/Università/AI nelle Imprese/open-data-analytics/data/raw...
reclassified_multiclass_aiuti_2015.csv - Record AI totali: 1, TIPO_AI assegnati: 1
reclassified_multiclass_aiuti_2016.csv - Record AI totali: 1, TIPO_AI assegnati: 1
reclassified_multiclass_aiuti_2014.csv - Record AI totali: 0, TIPO_AI assegnati: 0
reclassified_multiclass_aiuti_2017.csv - Record AI totali: 27, TIPO_AI assegnati: 27
reclassified_multiclass_aiuti_2019.csv - Record AI totali: 168, TIPO_AI assegnati: 168
reclassified_multiclass_aiuti_2018.csv - Record AI totali: 266, TIPO_AI assegnati: 266
reclassified_multiclass_aiuti_2025.csv - Record AI totali: 3053, TIPO_AI assegnati: 3053
reclassified_multiclass_aiuti_2022.csv - Record AI totali: 197, TIPO_AI assegnati: 197
reclassified_multiclass_aiuti_2020.csv - Record AI totali: 300, TIPO_AI assegnati: 300
reclassified_multiclass_aiuti_2021.csv - Record AI totali: 207, TIPO_AI assegnati: 207
reclassi